# REACT AGENT


In [1]:
import os
from dotenv import load_dotenv
import requests 
import json
from langchain_tavily import TavilySearch
import datetime
from openai import OpenAI

load_dotenv("apikey.env")

True

In [2]:
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
     print("Error: OPENAI_API_KEY environment variable not set")
url = "https://api.openai-proxy.org/v1"
client = OpenAI(api_key=api_key, base_url=url)

def llm(query, history=[], user_stop_words=[]):
    # 构建消息历史
    messages = [{'role': 'system', 'content': 'You are a helpful assistant.'}]
    for hist in history:
        messages.append({'role': 'user', 'content': hist[0]})
        messages.append({'role': 'assistant', 'content': hist[1]})
    messages.append({'role': 'user', 'content': query})
    
    try:
        resp = client.chat.completions.create(
            model="gpt-4.1",         
            messages=messages,
            temperature=0.1,             # 低温度确保确定性
            max_tokens=1024,
            stop=user_stop_words if user_stop_words else None
        )
        content = resp.choices[0].message.content
        return content
    
    except requests.exceptions.RequestException as e:
        return f"OPENAI API request error: {str(e)}"
    except (KeyError, IndexError, json.JSONDecodeError) as e:
        return f"OPENAI API response parsing error: {str(e)}"

In [3]:
# travily搜索引擎
# travily搜索引擎api key
travily = os.getenv("TAVILY-API-KEY")
if travily:
    os.environ['TAVILY_API_KEY']= travily
else:
    print("警告: 未设置 TAVILY-API-KEY 环境变量.")
tavily=TavilySearch(max_results=5)
tavily.description='这是一个类似谷歌和百度的搜索引擎，搜索知识、天气、股票、电影、小说、百科等都是支持的哦，如果你不确定就应该搜索一下，谢谢！s'

# 工具列表
tools=[tavily,]

tool_names=' or '.join([tool.name for tool in tools])  # 拼接工具名
tool_descs=[] # 拼接工具详情
for t in tools:
    args_desc=[]
    for name, info in t.args.items():
        args_desc.append({
            'name': name,
            'description': info.get('description', ''),
            'type': info.get('type', 'unknown')  # 或者 ''，或跳过
        })
tool_descs='\n'.join(tool_descs)

构造prompt --> `Thought/Action/Action Input/Observation`  

In [ ]:
prompt_tpl='''
    Today is {today}. Please Answer the following questions as best you can. You have access to the following tools:

    {tool_descs}

    These are chat history before:
    {chat_history}

    Use the following format:

    Question: the input question you must answer
    Thought: you should always think about what to do
    Action: the action to take, should be one of [{tool_names}]
    Action Input: the input to the action
    Observation: the result of the action
    ... (this Thought/Action/Action Input/Observation can be repeated zero or more times)
    Thought: I now know the final answer
    Final Answer: the final answer to the original input question

    Begin!

    Question: {query}
    {agent_scratchpad}
    '''

In [ ]:
def agent_execute(query,chat_history=[]):
    global tools,tool_names,tool_descs,prompt_tpl,llm
    
    agent_scratchpad='' # agent执行过程
    while True:
        # 1）触发llm思考下一步action
        history='\n'.join(['Question:%s\nAnswer:%s'%(his[0],his[1]) for his in chat_history])
        today=datetime.datetime.now().strftime('%Y-%m-%d')
        prompt=prompt_tpl.format(today=today, chat_history=history,
                                 tool_descs=tool_descs, tool_names=tool_names,
                                 query=query, agent_scratchpad=agent_scratchpad)
        
        print('\033[32m---等待LLM返回... ...\n%s\n\033[0m'%prompt,flush=True)
        response=llm(prompt,user_stop_words=['Observation:'])
        print('\033[34m---LLM返回---\n%s\n---\033[34m'%response,flush=True)
        
        # 2）解析thought+action+action input+observation or thought+final answer
        thought_i = response.rfind('Thought:')
        final_answer_i = response.rfind('\nFinal Answer:')
        action_i = response.rfind('\nAction:')
        action_input_i = response.rfind('\nAction Input:')
        observation_i = response.rfind('\nObservation:')
        
        # 3）返回final answer，执行完成
        if final_answer_i!=-1 and thought_i<final_answer_i:
            final_answer=response[final_answer_i+len('\nFinal Answer:'):].strip()
            chat_history.append((query,final_answer))
            return True, final_answer, chat_history
        
        # 4）解析action
        if not (thought_i<action_i<action_input_i):
            return False,'LLM回复格式异常',chat_history
        if observation_i==-1:
            observation_i=len(response)
            response=response+'Observation: '
        thought = response[thought_i+len('Thought:'):action_i].strip()
        action = response[action_i+len('\nAction:'):action_input_i].strip()
        action_input = response[action_input_i+len('\nAction Input:'):observation_i].strip()
        
        # 5）匹配tool
        the_tool=None
        for t in tools:
            if t.name==action:
                the_tool=t
                break
        if the_tool is None:
            observation='the tool not exist'
            agent_scratchpad=agent_scratchpad+response+observation+'\n'
            continue 
        
        # 6）执行tool
        try:
            action_input=json.loads(action_input)
        except json.JSONDecodeError:
            # 如果不是 JSON，就直接作为字符串构造
            action_input = {"query": action_input.strip()}
        try:    
            tool_ret=the_tool.invoke(action_input)
        except Exception as e:
            observation='the tool has error:{}'.format(e)
        else:
            observation=str(tool_ret)
        agent_scratchpad=agent_scratchpad+response+observation+'\n'

In [12]:
def agent_execute_with_retry(query,chat_history=[],retry_times=3):
    for i in range(retry_times):
        success, result, chat_history=agent_execute(query,chat_history=chat_history)
        if success:
            return success, result, chat_history
    return success, result, chat_history

In [13]:
my_history=[]
while True:
    query=input('query:')
    if query == "q":
        break
    success,result,my_history=agent_execute_with_retry(query,chat_history=my_history)
    my_history=my_history[-10:]

---等待LLM返回... ...

    Today is 2025-10-16. Please Answer the following questions as best you can. You have access to the following tools:

    

    These are chat history before:
    

    Use the following format:

    Question: the input question you must answer
    Thought: you should always think about what to do
    Action: the action to take, should be one of [tavily_search]
    Action Input: the input to the action
    Observation: the result of the action
    ... (this Thought/Action/Action Input/Observation can be repeated zero or more times)
    Thought: I now know the final answer
    Final Answer: the final answer to the original input question

    Begin!

    Question: 查询今年热门电影
    
    

---LLM返回---
Thought: 要回答关于今年（2025年）热门电影的问题，我需要查找2025年上映并且受欢迎的电影信息。
Action: tavily_search
Action Input: 2025年热门电影

---
---等待LLM返回... ...

    Today is 2025-10-16. Please Answer the following questions as best you can. You have access to the following tools:

    

    These are chat his